# 01 — Data Understanding, Preprocessing & Modeling Readiness

This notebook is the exploratory and data-preparation layer of the project. It keeps the workflow focused on data quality, leakage prevention, feature engineering, grouped splitting, and sanity checks.

The reusable implementation lives in `src/`, so the notebook does not duplicate the training pipeline.

**Target:** `glucose_120min`  
**Grouping variable:** `subject_id`


## 1. Imports and project configuration

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import GROUP_COLUMN, PROCESSED_DATA_PATH, RANDOM_STATE, RAW_DATA_PATH, TARGET, TEST_SIZE
from features import EXCLUDED_COLUMNS, QUALITY_RULES, add_prediction_features, clean_dataframe

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw dataset: {RAW_DATA_PATH}")
print(f"Target: {TARGET}")
print(f"Group column: {GROUP_COLUMN}")


## 2. Load the raw dataset

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())


## 3. Data audit

The first pass checks schema, missingness, duplicates, subject coverage, and target availability before any modeling operation.

In [ ]:
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique": df.nunique(dropna=False),
}).sort_values(["missing", "unique"], ascending=[False, True])

display(audit)

print(f"Duplicate rows: {df.duplicated().sum():,}")
print(f"Unique subjects: {df[GROUP_COLUMN].nunique(dropna=True):,}")
if "meal_id" in df.columns:
    print(f"Unique meals: {df['meal_id'].nunique(dropna=True):,}")
print(f"Missing target values: {df[TARGET].isna().sum():,}")


## 4. Target and feature distributions

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
summary = df[numeric_columns].describe().T

display(summary)

fig, ax = plt.subplots(figsize=(9, 5))
df[TARGET].dropna().plot(kind="hist", bins=30, ax=ax)
ax.set_title(f"Target Distribution: {TARGET}")
ax.set_xlabel(TARGET)
plt.tight_layout()
plt.show()


## 5. Missing-value investigation

In [ ]:
missing_summary = (
    df.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_pct=lambda x: (x["missing_count"] / len(df) * 100).round(2))
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
)

display(missing_summary)


## 6. Glucose consistency checks

Derived post-meal glucose delta columns are useful for validating the source data, but they are excluded from the predictive feature set because they depend on post-meal measurements.

In [ ]:
if {"premeal_glucose", "glucose_120min", "delta_glucose_120min"}.issubset(df.columns):
    expected_delta = df["glucose_120min"] - df["premeal_glucose"]
    observed_delta = df["delta_glucose_120min"]
    difference = (observed_delta - expected_delta).abs()
    print(f"120-minute delta max absolute difference: {difference.max()}")
    print(f"Rows with absolute difference > 0.001: {(difference > 0.001).sum():,}")

if {"premeal_glucose", "post_max_glucose_2h", "delta_glucose_max_2h"}.issubset(df.columns):
    expected_max_delta = df["post_max_glucose_2h"] - df["premeal_glucose"]
    observed_max_delta = df["delta_glucose_max_2h"]
    difference = (observed_max_delta - expected_max_delta).abs()
    print(f"Maximum-glucose delta max absolute difference: {difference.max()}")
    print(f"Rows with absolute difference > 0.001: {(difference > 0.001).sum():,}")


## 7. Define the prediction boundary

Only information available before or at meal time should be available to the model. The project therefore excludes:

- `meal_id` and `source_file` because they are identifiers or source metadata.
- `post_max_glucose_2h` because it is measured after the meal.
- `delta_glucose_max_2h` because it is derived from post-meal information.
- `delta_glucose_120min` because it directly contains the target information.

The raw `meal_time` timestamp is transformed into prediction-time calendar features rather than passed directly to the model.


In [ ]:
print("Excluded columns:")
for column in EXCLUDED_COLUMNS:
    print(f"- {column}")


## 8. Apply deterministic cleaning and feature engineering

The same functions are used by the training and inference scripts. This avoids notebook-only transformations that could silently differ from production inference.

In [ ]:
cleaned_df = clean_dataframe(df)
featured_df = add_prediction_features(cleaned_df)

print(f"Rows after deterministic quality filtering: {len(featured_df):,}")
display(featured_df.head())


## 9. Inspect engineered features

In [ ]:
engineered_columns = [
    "meal_day_of_week",
    "is_weekend",
    "carb_to_fiber_ratio",
    "total_macros",
    "carb_pct",
    "protein_pct",
    "fat_pct",
    "glucose_diff_baseline",
]

available_engineered = [column for column in engineered_columns if column in featured_df.columns]
display(featured_df[available_engineered].describe().T)


## 10. Build the modeling matrix

`subject_id` is retained separately for grouped splitting and is never exposed to the model. Missing values are handled inside the model pipeline so that imputation statistics are learned from training folds only.

In [ ]:
valid_target = featured_df[TARGET].notna()
model_df = featured_df.loc[valid_target].reset_index(drop=True)

y = model_df[TARGET].copy()
groups = model_df[GROUP_COLUMN].copy()
X = model_df.drop(columns=[TARGET, "meal_time", GROUP_COLUMN], errors="ignore")

categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_features = X.select_dtypes(include=np.number).columns.tolist()

print(f"Rows used for modeling: {len(X):,}")
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"Unique subjects: {groups.nunique():,}")


## 11. Subject-level train/test split

The same subject must not appear in both sets. This is the key safeguard against learning person-specific patterns that would inflate the reported test performance.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

train_subjects = set(groups.iloc[train_idx])
test_subjects = set(groups.iloc[test_idx])

print(f"Train rows: {len(train_idx):,}")
print(f"Test rows: {len(test_idx):,}")
print(f"Train subjects: {len(train_subjects):,}")
print(f"Test subjects: {len(test_subjects):,}")
print(f"Subject overlap: {len(train_subjects & test_subjects)}")


## 12. Leakage-safe preprocessing sanity check

The actual training script constructs this transformer inside a scikit-learn pipeline. This cell only verifies the feature groups used by that pipeline.

In [ ]:
from preprocessing import build_preprocessor

preprocessor = build_preprocessor(numeric_features, categorical_features)
print(preprocessor)


## 13. Save the processed dataset

The processed dataset is useful for inspection and reproducibility. The fitted preprocessing transformer is intentionally not saved from this notebook; the training script owns model fitting and artifact creation.

In [ ]:
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
featured_df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Saved: {PROCESSED_DATA_PATH}")


## 14. Final data checks

In [ ]:
checks = {
    "missing_target_after_filter": int(model_df[TARGET].isna().sum()),
    "duplicate_rows_after_cleaning": int(featured_df.duplicated().sum()),
    "subject_overlap_train_test": int(len(train_subjects & test_subjects)),
    "target_mean": float(y.mean()),
    "target_std": float(y.std()),
}

for name, value in checks.items():
    print(f"{name}: {value}")

assert checks["missing_target_after_filter"] == 0
assert checks["subject_overlap_train_test"] == 0


## 15. Next step

Run `python src/train.py` from the project root to compare the candidate models, tune Extra Trees when it is the best cross-validation candidate, evaluate the untouched held-out subjects, and generate row-level and subject-level error reports.